# DCCRN Full-Size Training on Colab (T4 GPU)

Trains the paper-scale DCCRN model (2.93M params, complex conv + complex LSTM,
STFT/ISTFT domain) on your defence-noise speech dataset, then evaluates it on
the test set (SNR, SI-SNR, STOI, PESQ) against the project targets
(SNR>15dB, STOI>0.85, PESQ>2.5).

**Before running:**
1. Zip your dataset so the zip contains this structure:
   ```
   dccrn_dataset/
     train/clean/*.wav       train/noisy/*.wav
     validation/clean/*.wav  validation/noisy/*.wav
     test/clean/*.wav        test/noisy/*.wav
   ```
2. Upload dccrn_dataset.zip to your Google Drive root (MyDrive/dccrn_dataset.zip).
   If you put it somewhere else, edit DRIVE_ZIP_PATH in the cell below.
3. Runtime -> Change runtime type -> GPU (T4) - you said this is already set.
4. Run all cells top to bottom (Runtime -> Run all).

Checkpoints, logs, and results are all saved to
MyDrive/dccrn_dataset_output/ on your Drive, so they survive a Colab
disconnect/timeout.

In [ ]:
!pip install -q soundfile pystoi pesq tqdm

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU not detected - check Runtime > Change runtime type > T4 GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

DRIVE_ZIP_PATH = "/content/drive/MyDrive/dccrn_dataset.zip"  # edit if your zip lives elsewhere
EXTRACT_ROOT = "/content/dccrn_dataset_extracted"
OUTPUT_DIR = "/content/drive/MyDrive/dccrn_dataset_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "results"), exist_ok=True)

if os.path.isdir(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)
os.makedirs(EXTRACT_ROOT, exist_ok=True)

# Extracted manually (not via extractall) because zips made on Windows
# (e.g. PowerShell's Compress-Archive) can store backslash path separators,
# which extractall() does not translate on Linux, silently flattening
# every file into one directory instead of raising an error.
print("Extracting dataset zip (backslash-safe manual extraction)...")
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    for info in zf.infolist():
        rel_parts = info.filename.replace("\\", "/").split("/")
        rel_parts = [p for p in rel_parts if p not in ("", ".", "..")]
        if not rel_parts:
            continue
        target_path = os.path.join(EXTRACT_ROOT, *rel_parts)
        if info.is_dir() or info.filename.endswith("/") or info.filename.endswith("\\"):
            os.makedirs(target_path, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        with zf.open(info) as src, open(target_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
print("Extraction done.")

DATA_ROOT = None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "train" in dirs and "test" in dirs and "validation" in dirs:
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    print("Could not auto-detect dataset root. Full extracted layout:")
    for root, dirs, files in os.walk(EXTRACT_ROOT):
        print(root, "dirs:", dirs, "files:", len(files))
else:
    print("DATA_ROOT =", DATA_ROOT)
    print("OUTPUT_DIR =", OUTPUT_DIR)
    for split in ["train", "validation", "test"]:
        clean_dir = os.path.join(DATA_ROOT, split, "clean")
        noisy_dir = os.path.join(DATA_ROOT, split, "noisy")
        print(split, "clean:", len(os.listdir(clean_dir)), "noisy:", len(os.listdir(noisy_dir)))

## Model: complex-domain DCCRN (encoder/decoder + complex LSTM)

In [ ]:
"""
DCCRN (Deep Complex Convolution Recurrent Network) for speech enhancement.

Reference: Hu et al., "DCCRN: Deep Complex Convolution Recurrent Network for
Phase-Aware Speech Enhancement", Interspeech 2020.

The network operates directly on the complex STFT of the noisy waveform.
A complex-valued U-Net (complex conv encoder + complex conv decoder with skip
connections) with a complex LSTM bottleneck predicts a complex ratio mask
(bounded with tanh), which is applied to the noisy spectrogram. The masked
spectrogram is inverted back to the time domain with ISTFT and trained with
an SI-SNR loss, so phase is preserved and no separate phase estimator is
needed.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Complex building blocks
# ---------------------------------------------------------------------------
class ComplexConv2d(nn.Module):
    """Complex 2D convolution implemented with two real convolutions.

    (a + ib) * (c + id) = (ac - bd) + i(ad + bc)
    """

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.real_conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding)
        self.imag_conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding)
        nn.init.xavier_uniform_(self.real_conv.weight)
        nn.init.xavier_uniform_(self.imag_conv.weight)
        nn.init.zeros_(self.real_conv.bias)
        nn.init.zeros_(self.imag_conv.bias)

    def forward(self, x_r, x_i):
        real = self.real_conv(x_r) - self.imag_conv(x_i)
        imag = self.real_conv(x_i) + self.imag_conv(x_r)
        return real, imag


class ComplexConvTranspose2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, output_padding=0):
        super().__init__()
        self.real_conv = nn.ConvTranspose2d(
            in_ch, out_ch, kernel_size, stride, padding, output_padding=output_padding
        )
        self.imag_conv = nn.ConvTranspose2d(
            in_ch, out_ch, kernel_size, stride, padding, output_padding=output_padding
        )
        nn.init.xavier_uniform_(self.real_conv.weight)
        nn.init.xavier_uniform_(self.imag_conv.weight)
        nn.init.zeros_(self.real_conv.bias)
        nn.init.zeros_(self.imag_conv.bias)

    def forward(self, x_r, x_i):
        real = self.real_conv(x_r) - self.imag_conv(x_i)
        imag = self.real_conv(x_i) + self.imag_conv(x_r)
        return real, imag


class ComplexBatchNorm2d(nn.Module):
    """Naive complex batch-norm: independent BN on real/imag streams."""

    def __init__(self, num_features):
        super().__init__()
        self.bn_r = nn.BatchNorm2d(num_features)
        self.bn_i = nn.BatchNorm2d(num_features)

    def forward(self, x_r, x_i):
        return self.bn_r(x_r), self.bn_i(x_i)


class ComplexPReLU(nn.Module):
    def __init__(self, num_parameters=1):
        super().__init__()
        self.act_r = nn.PReLU(num_parameters)
        self.act_i = nn.PReLU(num_parameters)

    def forward(self, x_r, x_i):
        return self.act_r(x_r), self.act_i(x_i)


class ComplexLSTM(nn.Module):
    """Complex LSTM built from real LSTMs following complex multiplication
    rules for the linear projections. Operates on (B, T, F) real/imag pairs.
    """

    def __init__(self, input_size, hidden_size, num_layers=2, bidirectional=False):
        super().__init__()
        self.lstm_r = nn.LSTM(
            input_size, hidden_size, num_layers=num_layers,
            batch_first=True, bidirectional=bidirectional,
        )
        self.lstm_i = nn.LSTM(
            input_size, hidden_size, num_layers=num_layers,
            batch_first=True, bidirectional=bidirectional,
        )
        out_mul = 2 if bidirectional else 1
        self.hidden_size = hidden_size * out_mul

    def forward(self, x_r, x_i):
        # F_rr = LSTM_r(real), F_ir = LSTM_r(imag) etc. Approximate complex
        # matmul by running each real LSTM on both streams and combining.
        r2r, _ = self.lstm_r(x_r)
        r2i, _ = self.lstm_i(x_r)
        i2r, _ = self.lstm_r(x_i)
        i2i, _ = self.lstm_i(x_i)
        real = r2r - i2i
        imag = r2i + i2r
        return real, imag


# ---------------------------------------------------------------------------
# DCCRN model
# ---------------------------------------------------------------------------
class DCCRN(nn.Module):
    def __init__(
        self,
        n_fft=512,
        hop_length=100,
        win_length=400,
        channels=(16, 32, 64, 64, 128, 128),
        kernel_size=(5, 2),
        stride=(2, 1),
        lstm_hidden=128,
        lstm_layers=2,
        masking_mode="E",  # "E" = complex ratio mask, "R" = direct mapping
    ):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.masking_mode = masking_mode
        self.register_buffer("window", torch.hann_window(win_length), persistent=False)

        ch = [2] + list(channels)  # input has 1 complex channel -> real/imag split into 2 "features" but we keep 1 channel complex pair
        ch[0] = 1
        pad_f = (kernel_size[0] - 1) // 2

        self.encoders = nn.ModuleList()
        self.enc_bns = nn.ModuleList()
        self.enc_acts = nn.ModuleList()
        for i in range(len(channels)):
            self.encoders.append(
                ComplexConv2d(ch[i], ch[i + 1], kernel_size, stride, padding=(pad_f, 0))
            )
            self.enc_bns.append(ComplexBatchNorm2d(ch[i + 1]))
            self.enc_acts.append(ComplexPReLU())

        # Compute frequency dimension after the encoder stack to size the LSTM.
        # The DC bin is dropped before the encoder (see forward()), so the
        # encoder sees n_fft // 2 frequency bins, not n_fft // 2 + 1.
        freq_dim = n_fft // 2
        for _ in channels:
            freq_dim = (freq_dim + 2 * pad_f - kernel_size[0]) // stride[0] + 1
        self.freq_dim = freq_dim
        lstm_input = freq_dim * channels[-1]

        self.lstm = ComplexLSTM(lstm_input, lstm_hidden, num_layers=lstm_layers)
        self.lstm_proj_r = nn.Linear(self.lstm.hidden_size, lstm_input)
        self.lstm_proj_i = nn.Linear(self.lstm.hidden_size, lstm_input)

        dec_channels = list(reversed(channels)) + [1]
        self.decoders = nn.ModuleList()
        self.dec_bns = nn.ModuleList()
        self.dec_acts = nn.ModuleList()
        for i in range(len(channels)):
            in_ch = dec_channels[i] * 2  # skip connection concat (real&imag each doubled channel-wise)
            out_ch = dec_channels[i + 1]
            last = i == len(channels) - 1
            self.decoders.append(
                ComplexConvTranspose2d(
                    in_ch, out_ch, kernel_size, stride,
                    padding=(pad_f, 0), output_padding=(stride[0] - 1, 0),
                )
            )
            if not last:
                self.dec_bns.append(ComplexBatchNorm2d(out_ch))
                self.dec_acts.append(ComplexPReLU())
            else:
                self.dec_bns.append(None)
                self.dec_acts.append(None)

    # --------------------------- STFT helpers ---------------------------
    def stft(self, wav):
        spec = torch.stft(
            wav, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length,
            window=self.window, return_complex=True, center=True,
        )
        return spec.real, spec.imag

    def istft(self, real, imag, length=None):
        spec = torch.complex(real, imag)
        wav = torch.istft(
            spec, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length,
            window=self.window, center=True, length=length,
        )
        return wav

    # ----------------------------- forward -------------------------------
    def forward(self, noisy_wav):
        """noisy_wav: (B, T) float tensor in [-1, 1]. Returns enhanced wav (B, T)."""
        length = noisy_wav.shape[-1]
        spec_r, spec_i = self.stft(noisy_wav)  # (B, F, Tf)
        x_r = spec_r.unsqueeze(1)  # (B, 1, F, Tf)
        x_i = spec_i.unsqueeze(1)

        # drop the Nyquist bin so freq dim is even/tidy for strided convs
        x_r = x_r[:, :, 1:, :]
        x_i = x_i[:, :, 1:, :]

        skips = []
        for conv, bn, act in zip(self.encoders, self.enc_bns, self.enc_acts):
            x_r, x_i = conv(x_r, x_i)
            x_r, x_i = bn(x_r, x_i)
            x_r, x_i = act(x_r, x_i)
            skips.append((x_r, x_i))

        B, C, Fq, Tf = x_r.shape
        seq_r = x_r.permute(0, 3, 1, 2).reshape(B, Tf, C * Fq)
        seq_i = x_i.permute(0, 3, 1, 2).reshape(B, Tf, C * Fq)
        h_r, h_i = self.lstm(seq_r, seq_i)
        proj_r = self.lstm_proj_r(h_r) - self.lstm_proj_i(h_i)
        proj_i = self.lstm_proj_r(h_i) + self.lstm_proj_i(h_r)
        x_r = proj_r.reshape(B, Tf, C, Fq).permute(0, 2, 3, 1)
        x_i = proj_i.reshape(B, Tf, C, Fq).permute(0, 2, 3, 1)

        for idx, (deconv, bn, act) in enumerate(zip(self.decoders, self.dec_bns, self.dec_acts)):
            skip_r, skip_i = skips[-(idx + 1)]
            x_r = torch.cat([x_r, skip_r], dim=1)
            x_i = torch.cat([x_i, skip_i], dim=1)
            x_r, x_i = deconv(x_r, x_i)
            if bn is not None:
                x_r, x_i = bn(x_r, x_i)
                x_r, x_i = act(x_r, x_i)

        mask_r = x_r.squeeze(1)  # (B, F-1, Tf)
        mask_i = x_i.squeeze(1)

        # pad back the Nyquist bin (mask = 1, i.e. pass-through) that we dropped earlier
        pad = torch.zeros_like(mask_r[:, :1, :])
        mask_r = torch.cat([pad, mask_r], dim=1)
        mask_i = torch.cat([pad, mask_i], dim=1)

        if self.masking_mode == "E":
            mask_r = torch.tanh(mask_r)
            mask_i = torch.tanh(mask_i)
            enh_r = spec_r * mask_r - spec_i * mask_i
            enh_i = spec_r * mask_i + spec_i * mask_r
        else:  # direct complex spectral mapping
            enh_r, enh_i = mask_r, mask_i

        enhanced = self.istft(enh_r, enh_i, length=length)
        return enhanced

## Dataset loader (parses noise type / SNR from filenames)

In [ ]:
"""Paired clean/noisy speech dataset for DCCRN training.

Expects the layout produced by the dccrn_dataset generator:

    <root>/train/clean/*.wav       <root>/train/noisy/*.wav
    <root>/validation/clean/*.wav  <root>/validation/noisy/*.wav
    <root>/test/clean/*.wav        <root>/test/noisy/*.wav

with identical filenames in the clean/noisy pair, e.g.
"1272-128104-0000__engine__+20dB__000000.wav". The noise type and SNR are
encoded in the filename (speaker-utt__noisetype__SNRdB__index.wav) and are
parsed out for per-condition evaluation breakdowns.
"""
import os
import random
import re
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from torch.utils.data import Dataset

FNAME_RE = re.compile(r"^(?P<utt>.+)__(?P<noise>[^_]+)__(?P<snr>[+-]?\d+)dB__(?P<idx>\d+)$")


def parse_filename(path):
    stem = Path(path).stem
    m = FNAME_RE.match(stem)
    if m is None:
        return {"utt": stem, "noise": "unknown", "snr": None}
    return {"utt": m.group("utt"), "noise": m.group("noise"), "snr": int(m.group("snr"))}


class NoisyCleanDataset(Dataset):
    def __init__(self, root, split, sample_rate=16000, segment_seconds=4.0, train=True):
        self.clean_dir = os.path.join(root, split, "clean")
        self.noisy_dir = os.path.join(root, split, "noisy")
        self.files = sorted(
            f for f in os.listdir(self.clean_dir) if f.lower().endswith(".wav")
        )
        if not self.files:
            raise RuntimeError(f"No wav files found in {self.clean_dir}")
        self.sample_rate = sample_rate
        self.segment_len = int(segment_seconds * sample_rate)
        self.train = train

    def __len__(self):
        return len(self.files)

    def _load(self, path):
        wav, sr = sf.read(path, dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != self.sample_rate:
            raise ValueError(f"{path} has sample rate {sr}, expected {self.sample_rate}")
        return wav

    def __getitem__(self, idx):
        fname = self.files[idx]
        clean = self._load(os.path.join(self.clean_dir, fname))
        noisy = self._load(os.path.join(self.noisy_dir, fname))

        n = min(len(clean), len(noisy))
        clean, noisy = clean[:n], noisy[:n]

        if self.train:
            if n >= self.segment_len:
                start = random.randint(0, n - self.segment_len)
                clean = clean[start:start + self.segment_len]
                noisy = noisy[start:start + self.segment_len]
            else:
                pad = self.segment_len - n
                clean = np.pad(clean, (0, pad))
                noisy = np.pad(noisy, (0, pad))

        meta = parse_filename(fname)
        return (
            torch.from_numpy(noisy).float(),
            torch.from_numpy(clean).float(),
            fname,
            meta,
        )


def collate_eval(batch):
    """Keep variable-length items as a list for full-utterance evaluation."""
    noisy = [b[0] for b in batch]
    clean = [b[1] for b in batch]
    fnames = [b[2] for b in batch]
    metas = [b[3] for b in batch]
    return noisy, clean, fnames, metas

## Loss functions (SI-SNR + multi-resolution STFT)

In [ ]:
"""Loss functions for time-domain speech enhancement training."""
import torch


def si_snr(estimate, target, eps=1e-8):
    """Scale-invariant SNR (higher is better), computed per-item in the batch.

    estimate, target: (B, T)
    returns: (B,) SI-SNR in dB
    """
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)
    target = target - target.mean(dim=-1, keepdim=True)

    s_target = (torch.sum(estimate * target, dim=-1, keepdim=True) /
                (torch.sum(target * target, dim=-1, keepdim=True) + eps)) * target
    e_noise = estimate - s_target

    ratio = torch.sum(s_target ** 2, dim=-1) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)


def si_snr_loss(estimate, target):
    """Negative mean SI-SNR, suitable for minimization."""
    return -si_snr(estimate, target).mean()


def multi_resolution_stft_loss(estimate, target, fft_sizes=(512, 1024, 2048),
                                hop_sizes=(100, 200, 400), win_sizes=(400, 800, 1600)):
    """Auxiliary perceptual-ish loss: L1 on log-magnitude spectrograms at
    multiple resolutions, encouraging spectral detail beyond time-domain SI-SNR.
    """
    loss = 0.0
    for n_fft, hop, win in zip(fft_sizes, hop_sizes, win_sizes):
        window = torch.hann_window(win, device=estimate.device)
        est_spec = torch.stft(estimate, n_fft=n_fft, hop_length=hop, win_length=win,
                               window=window, return_complex=True, center=True)
        tgt_spec = torch.stft(target, n_fft=n_fft, hop_length=hop, win_length=win,
                               window=window, return_complex=True, center=True)
        est_mag = torch.log(torch.abs(est_spec) + 1e-5)
        tgt_mag = torch.log(torch.abs(tgt_spec) + 1e-5)
        loss = loss + torch.nn.functional.l1_loss(est_mag, tgt_mag)
    return loss / len(fft_sizes)


def combined_loss(estimate, target, stft_weight=0.2):
    """SI-SNR (primary, matches DCCRN's original training objective) plus a
    smaller multi-resolution log-STFT term for extra spectral sharpness."""
    return si_snr_loss(estimate, target) + stft_weight * multi_resolution_stft_loss(estimate, target)

## Evaluation metrics (SNR, SI-SNR, STOI, PESQ)

In [ ]:
"""Evaluation metrics: SNR, STOI, PESQ, SI-SNR â€” all on numpy arrays at 16 kHz."""
import numpy as np
from pesq import pesq as pesq_fn
from pystoi import stoi as stoi_fn


def snr(estimate, target, eps=1e-8):
    noise = estimate - target
    return 10 * np.log10((np.sum(target ** 2) + eps) / (np.sum(noise ** 2) + eps))


def si_snr_np(estimate, target, eps=1e-8):
    estimate = estimate - estimate.mean()
    target = target - target.mean()
    s_target = (np.sum(estimate * target) / (np.sum(target ** 2) + eps)) * target
    e_noise = estimate - s_target
    return 10 * np.log10((np.sum(s_target ** 2) + eps) / (np.sum(e_noise ** 2) + eps))


def stoi_score(estimate, target, sr=16000):
    return stoi_fn(target, estimate, sr, extended=False)


def pesq_score(estimate, target, sr=16000):
    mode = "wb" if sr == 16000 else "nb"
    try:
        return pesq_fn(sr, target, estimate, mode)
    except Exception:
        return float("nan")


def evaluate_pair(estimate, target, sr=16000):
    n = min(len(estimate), len(target))
    estimate, target = estimate[:n], target[:n]
    return {
        "snr": snr(estimate, target),
        "si_snr": si_snr_np(estimate, target),
        "stoi": stoi_score(estimate, target, sr),
        "pesq": pesq_score(estimate, target, sr),
    }

## Training (full paper-scale config, GPU)

This is the config we skipped locally because it needed ~56 min/epoch on
CPU. On a T4 this should be dramatically faster - watch the first epoch's
timing print and adjust EPOCHS if you want to shorten/extend the run.

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import time

# ---- Full paper-scale hyperparameters (GPU) ----
EPOCHS = 60
BATCH_SIZE = 16
LR = 1e-3
SEGMENT_SECONDS = 4.0
SAMPLE_RATE = 16000
CHANNELS = (16, 32, 64, 64, 128, 128)
LSTM_HIDDEN = 128
LSTM_LAYERS = 2
STFT_LOSS_WEIGHT = 0.2
GRAD_CLIP = 5.0
NUM_WORKERS = 2

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

train_set = NoisyCleanDataset(DATA_ROOT, "train", SAMPLE_RATE, SEGMENT_SECONDS, train=True)
val_set = NoisyCleanDataset(DATA_ROOT, "validation", SAMPLE_RATE, SEGMENT_SECONDS, train=True)
print(f"Train utterances: {len(train_set)} | Validation utterances: {len(val_set)}")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

model = DCCRN(n_fft=512, hop_length=100, win_length=400, channels=CHANNELS,
              lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
writer = SummaryWriter(LOG_DIR)

best_val_loss = float("inf")
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_start = time.time()
    running_loss, n_batches = 0.0, 0
    for i, (noisy, clean, _, _) in enumerate(train_loader):
        noisy, clean = noisy.to(device), clean.to(device)
        estimate = model(noisy)
        estimate = estimate[..., :clean.shape[-1]]
        loss = combined_loss(estimate, clean, stft_weight=STFT_LOSS_WEIGHT)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        running_loss += loss.item(); n_batches += 1; global_step += 1
        writer.add_scalar("train/loss_step", loss.item(), global_step)
        if i % 20 == 0:
            print(f"epoch {epoch} batch {i}/{len(train_loader)} loss {loss.item():.4f}")
    train_loss = running_loss / max(n_batches, 1)

    model.eval()
    val_loss, val_sisnr, n_val = 0.0, 0.0, 0
    with torch.no_grad():
        for noisy, clean, _, _ in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            estimate = model(noisy)
            estimate = estimate[..., :clean.shape[-1]]
            loss = combined_loss(estimate, clean, stft_weight=STFT_LOSS_WEIGHT)
            val_loss += loss.item()
            val_sisnr += si_snr(estimate, clean).mean().item()
            n_val += 1
    val_loss /= max(n_val, 1); val_sisnr /= max(n_val, 1)
    scheduler.step(val_loss)

    elapsed = time.time() - epoch_start
    print(f"== epoch {epoch} done in {elapsed:.1f}s | train_loss {train_loss:.4f} "
          f"| val_loss {val_loss:.4f} | val_SI-SNR {val_sisnr:.2f} dB ==")
    writer.add_scalar("train/loss_epoch", train_loss, epoch)
    writer.add_scalar("val/loss_epoch", val_loss, epoch)
    writer.add_scalar("val/si_snr", val_sisnr, epoch)

    ckpt = {
        "epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
        "val_loss": val_loss,
        "args": {"channels": ",".join(map(str, CHANNELS)), "lstm_hidden": LSTM_HIDDEN, "lstm_layers": LSTM_LAYERS},
    }
    torch.save(ckpt, os.path.join(CHECKPOINT_DIR, "last.pt"))
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(ckpt, os.path.join(CHECKPOINT_DIR, "best.pt"))
        print(f"  -> new best model saved (val_loss={val_loss:.4f})")

writer.close()
print("Training complete. Checkpoints saved to", CHECKPOINT_DIR)

## Evaluate on the test set (same metrics/breakdowns as the local run)

In [ ]:
import numpy as np, collections, csv

ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best.pt"), map_location=device)
train_args = ckpt.get("args", {})
channels = tuple(int(c) for c in train_args.get("channels", "16,32,64,64,128,128").split(","))
eval_model = DCCRN(n_fft=512, hop_length=100, win_length=400, channels=channels,
                    lstm_hidden=train_args.get("lstm_hidden", 128),
                    lstm_layers=train_args.get("lstm_layers", 2)).to(device)
eval_model.load_state_dict(ckpt["model_state"])
eval_model.eval()
print(f"Loaded checkpoint from epoch {ckpt.get('epoch')}")

test_set = NoisyCleanDataset(DATA_ROOT, "test", SAMPLE_RATE, train=False)
samples_dir = os.path.join(OUTPUT_DIR, "results", "enhanced_samples")
os.makedirs(samples_dir, exist_ok=True)

rows = []
saved = 0
with torch.no_grad():
    for idx in range(len(test_set)):
        noisy, clean, fname, meta = test_set[idx]
        estimate = eval_model(noisy.unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        clean_np, noisy_np = clean.numpy(), noisy.numpy()
        n = min(len(estimate), len(clean_np), len(noisy_np))
        estimate, clean_np, noisy_np = estimate[:n], clean_np[:n], noisy_np[:n]

        noisy_metrics = evaluate_pair(noisy_np, clean_np, SAMPLE_RATE)
        enh_metrics = evaluate_pair(estimate, clean_np, SAMPLE_RATE)
        rows.append({
            "file": fname, "noise_type": meta["noise"], "input_snr_db": meta["snr"],
            "noisy_snr": noisy_metrics["snr"], "enh_snr": enh_metrics["snr"],
            "noisy_si_snr": noisy_metrics["si_snr"], "enh_si_snr": enh_metrics["si_snr"],
            "noisy_stoi": noisy_metrics["stoi"], "enh_stoi": enh_metrics["stoi"],
            "noisy_pesq": noisy_metrics["pesq"], "enh_pesq": enh_metrics["pesq"],
        })
        if saved < 10:
            sf.write(os.path.join(samples_dir, f"enhanced_{fname}"), estimate, SAMPLE_RATE)
            sf.write(os.path.join(samples_dir, f"noisy_{fname}"), noisy_np, SAMPLE_RATE)
            sf.write(os.path.join(samples_dir, f"clean_{fname}"), clean_np, SAMPLE_RATE)
            saved += 1
        if idx % 50 == 0:
            print(f"evaluated {idx}/{len(test_set)}")

csv_path = os.path.join(OUTPUT_DIR, "results", "test_metrics.csv")
with open(csv_path, "w", newline="") as f:
    writer_csv = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer_csv.writeheader(); writer_csv.writerows(rows)
print("Saved per-file metrics to", csv_path)

def mean(key, source=rows):
    vals = [r[key] for r in source if not np.isnan(r[key])]
    return float(np.mean(vals)) if vals else float("nan")

print(f"\n===== Overall test-set results (mean over {len(rows)} utterances) =====")
print(f"{'Metric':<10}{'Noisy':>10}{'Enhanced':>10}{'Delta':>10}")
for label, nk, ek in [("SNR (dB)","noisy_snr","enh_snr"), ("SI-SNR","noisy_si_snr","enh_si_snr"),
                      ("STOI","noisy_stoi","enh_stoi"), ("PESQ","noisy_pesq","enh_pesq")]:
    nv, ev = mean(nk), mean(ek)
    print(f"{label:<10}{nv:>10.3f}{ev:>10.3f}{ev-nv:>+10.3f}")

print("\n===== Target spec check =====")
for k, v in {"SNR > 15 dB": mean("enh_snr")>15, "STOI > 0.85": mean("enh_stoi")>0.85,
             "PESQ > 2.5": mean("enh_pesq")>2.5}.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")

print("\n===== Breakdown by noise type =====")
by_noise = collections.defaultdict(list)
for r in rows: by_noise[r["noise_type"]].append(r)
for noise, group in sorted(by_noise.items()):
    print(f"{noise:<12} n={len(group):<5} PESQ={mean('enh_pesq',group):.2f}  "
          f"STOI={mean('enh_stoi',group):.3f}  SNR={mean('enh_snr',group):.2f} dB")

print("\n===== Breakdown by input SNR level =====")
by_snr = collections.defaultdict(list)
for r in rows: by_snr[r["input_snr_db"]].append(r)
for snr_level, group in sorted(by_snr.items(), key=lambda kv: (kv[0] is None, kv[0])):
    print(f"{snr_level}dB{'':<8} n={len(group):<5} PESQ={mean('enh_pesq',group):.2f}  "
          f"STOI={mean('enh_stoi',group):.3f}  SNR={mean('enh_snr',group):.2f} dB")

print(f"\nEnhanced/noisy/clean samples saved to {samples_dir}")
print(f"Checkpoints saved to {CHECKPOINT_DIR}")
print("\nDownload checkpoints/best.pt from your Drive when you want to deploy it "
      "with infer.py (same file as the CPU/Pi/Jetson deployment path).")